In [1]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append(r"Z:\HTOC\Data_Analytics\threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    display(f"Loaded config from: {config_path}")
    display(f"Base URL: {api_base_url}")
    display(f"Access ID: {api_access_id}")
    display(f"Default Org: {api_default_org}")
except Exception as e:
    display(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    display("ThreatConnect initialized.")
except Exception as e:
    display(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    display("RequestObject successfully created.")
except Exception as e:
    display(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)




'Loaded config from: C:\\Users\\jaskew\\Documents\\project_repository\\scripts\\Data Movement\\ThrearConnect-api-pull\\utils\\config.json'

'Base URL: https://hvs.threatconnect.com/api'

'Access ID: 09783848890162390382'

'Default Org: HTOC Org'

'ThreatConnect initialized.'

'RequestObject successfully created.'

In [2]:
import pandas as pd
from datetime import datetime, timedelta
import pytz
import urllib.parse

# Configuration for ThreatConnect indicator query
QUERY_LOOKBACK_DAYS = 1  # days of lastObserved activity to include
INDICATOR_TYPE_NAMES = [
    "Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR",
    "Email Subject", "Hashtag", "Mutex", "Registry Key", "User Agent",
]
OWNER_NAMES = [
    'HTOC Org',
    'CISA Federal Feed',
    'CMS_CTI',
    'Crowdstrike Falcon Intelligence',
    'DHS CISCP',
    'Intel471',
    'Mandiant Advantage Threat Intelligence',
    'VA_TIP Data',
]
RESULT_PAGE_SIZE = 500  # keep this smaller; same fields, just paged

# Setup
cutoff = pd.Timestamp.utcnow()
start_date = (datetime.now(pytz.UTC) - timedelta(days=QUERY_LOOKBACK_DAYS)).date()
start = f"{start_date}T00:00:00Z"

type_names = INDICATOR_TYPE_NAMES
type_name_condition = ", ".join([f'"{t}"' for t in type_names])

list_of_owners = OWNER_NAMES

# Build owner IN (...) clause
owner_condition = ", ".join([f'"{o}"' for o in list_of_owners])

tql_raw = (
    f'ownerName IN ({owner_condition}) AND '
    f'typeName IN ({type_name_condition}) AND '
    f'lastObserved >= "{start}"'
)

tql_encoded = urllib.parse.quote(tql_raw)

final_results = []

# Query indicators (paginate so you don't 502 with heavy fields)
# Create a NEW RequestObject WITHOUT owner restriction to query across all owners
ro_multi = RequestObject()
ro_multi.set_http_method('GET')

result_start = 0
result_limit = RESULT_PAGE_SIZE

while True:
    try:
        # NOTE: same fields list you requested (tags,observations,associatedGroups,falsePositives,threatAssess)
        # Only change here is removing the trailing comma after threatAssess which can break parsing.
        ro_multi.set_request_uri(
            f'/v3/indicators?tql={tql_encoded}'
            f'&fields=tags,observations,associatedGroups,falsePositives,threatAssess'
            f'&resultStart={result_start}&resultLimit={result_limit}'
        )

        response = tc.api_request(ro_multi)

        ct = response.headers.get('content-type', '')
        if not ct.startswith('application/json'):
            raise RuntimeError(f"Non-JSON response ({ct}): {response.content[:200]}")

        results = response.json()
        data_items = results.get('data', []) or []

        # stop when no more results
        if not data_items:
            break

        final_results.append(results)
        result_start += result_limit

    except Exception as e:
        display(f"Failed to query indicators (start={result_start}): {e}")
        break

# Normalize results
normalized_data = []
for result in final_results:
    data_items = result.get('data', [])
    if not data_items:
        display("No data returned in API response:", result)
    for item in data_items:
        if isinstance(item, dict) and 'summary' in item:
            normalized_data.append(item)

if normalized_data:
    observed_src = pd.json_normalize(normalized_data)
    observed_src['indicator'] = observed_src['summary'].astype(str).str.split().str[0].str.strip()
    observed_src['lastObserved'] = pd.to_datetime(observed_src['lastObserved'], utc=True, errors='coerce')
    observed_src = observed_src[observed_src['lastObserved'] >= pd.to_datetime(start, utc=True)]
    
    # Create a 'sources' column by aggregating ownerName values per indicator
    sources_per_indicator = (
        observed_src.groupby('indicator')['ownerName']
        .apply(lambda x: ', '.join(sorted(set(x))))
        .reset_index()
        .rename(columns={'ownerName': 'sources'})
    )

    # Merge sources back into observed_src
    observed_src = observed_src.merge(sources_per_indicator, on='indicator', how='left')
    # Filter to keep only records where ownerName is 'HTOC Org'
    observed_src = observed_src[observed_src['ownerName'] == 'HTOC Org'].copy()
else:
    display("No valid indicator data found.")
    observed_src = pd.DataFrame()

display(observed_src)

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,threatAssessRating,...,legacyLink,tags.data,associatedGroups.data,source,hostName,dnsActive,whoisActive,address,indicator,sources
0,5629499555060709,2025-06-16T17:42:41Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-03-20T12:12:03Z,3.0,52.0,3.00,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 35760, 'name': 'OS Splunk API', 'descr...",NaN,NaN,NaN,NaN,NaN,NaN,72.5.43.62,HTOC Org
1,5629499555105171,2025-06-25T17:45:49Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-03-20T12:11:49Z,3.0,71.0,3.00,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 2023876, 'name': 'VA OIT CSOC CTS Splu...",NaN,NaN,NaN,NaN,NaN,NaN,64.62.156.157,HTOC Org
2,5629499563360167,2025-08-13T15:08:34Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-03-20T12:11:36Z,3.0,51.0,3.00,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 2023876, 'name': 'VA OIT CSOC CTS Splu...",NaN,NaN,NaN,NaN,NaN,NaN,65.49.1.42,HTOC Org
3,5629499566213979,2025-09-04T14:20:28Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-03-20T12:11:21Z,3.0,55.0,1.50,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 2023876, 'name': 'VA OIT CSOC CTS Splu...",NaN,NaN,NaN,NaN,NaN,NaN,121.159.71.249,HTOC Org
4,5629499578170403,2025-11-26T17:40:37Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-03-20T12:11:07Z,3.0,66.0,2.67,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 2023876, 'name': 'VA OIT CSOC CTS Splu...",NaN,NaN,NaN,NaN,NaN,NaN,167.94.138.176,HTOC Org
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1186,6755399465229332,2025-07-28T17:15:57Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-03-13T00:23:53Z,3.0,48.0,1.00,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...",NaN,NaN,NaN,NaN,NaN,NaN,205.210.31.66,"DHS CISCP, HTOC Org"
1187,6755399465229545,2025-07-28T17:34:27Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-03-13T00:23:50Z,3.0,48.0,1.50,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...",NaN,NaN,NaN,NaN,NaN,NaN,198.235.24.109,"DHS CISCP, HTOC Org"
1188,5629499556005867,2025-06-30T12:21:33Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-03-13T00:23:50Z,3.0,49.0,3.00,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...",NaN,NaN,NaN,NaN,NaN,NaN,65.49.1.183,HTOC Org
1189,4403670,2023-09-18T15:47:32Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-03-13T00:23:46Z,5.0,96.0,1.71,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 2023876, 'name': 'VA OIT CSOC CTS Splu...","[{'id': 6755399482040879, 'dateAdded': '2026-0...",NaN,NaN,NaN,NaN,NaN,198.54.117.242,"CISA Federal Feed, HTOC Org"


In [3]:
import pandas as pd
import ast

# Load the Excel file
file_path = r"Z:\HTOC\Data_Analytics\Data\Threat Assessment Scores\Threat_Assessment_Scores.xlsx"
df = pd.read_excel(file_path)

# Keep only indicators that are also in observed_src
_indicator_col = next((c for c in ["indicator", "Indicator", "INDICATOR"] if c in df.columns), None)
if _indicator_col is None:
    raise KeyError(f"Could not find indicator column in df. Columns: {list(df.columns)}")

_observed_indicators = set(observed_src["indicator"].dropna().astype(str))
df = df[df[_indicator_col].astype(str).isin(_observed_indicators)].copy()

# Update df's Last Observed from observed_src.lastObserved
_last_observed_col = next(
    (
        c
        for c in [
            "Last Observed",
            "lastObserved",
            "LastObserved",
            "last_observed",
            "LAST OBSERVED",
        ]
        if c in df.columns
    ),
    None,
)
if _last_observed_col is None:
    raise KeyError(f"Could not find 'Last Observed' column in df. Columns: {list(df.columns)}")

_assoc_groups_src_col = "associatedGroups.data"
_assoc_groups_target_col = "Associated Groups"
if _assoc_groups_src_col not in observed_src.columns:
    raise KeyError(
        f"Could not find '{_assoc_groups_src_col}' column in observed_src. Columns: {list(observed_src.columns)}"
    )


def _extract_group_ids(value):
    # Handle scalar nulls safely; avoid pd.isna on list-like values.
    if value is None:
        return pd.NA

    parsed = value
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return pd.NA
        try:
            parsed = ast.literal_eval(text)
        except (ValueError, SyntaxError):
            return text
    elif isinstance(value, float) and pd.isna(value):
        return pd.NA

    if isinstance(parsed, dict):
        gid = parsed.get("id")
        return f"Group Id: {gid}" if gid is not None else pd.NA

    if isinstance(parsed, list):
        ids = []
        for item in parsed:
            if isinstance(item, dict) and item.get("id") is not None:
                ids.append(f"Group Id: {item.get('id')}")
        return ", ".join(ids) if ids else pd.NA

    return pd.NA


_observed_latest = (
    observed_src.dropna(subset=["indicator"])
    .assign(
        indicator=lambda d: d["indicator"].astype(str),
        lastObserved=lambda d: pd.to_datetime(d["lastObserved"], utc=True, errors="coerce"),
    )
    .sort_values("lastObserved")
    .drop_duplicates(subset=["indicator"], keep="last")
)

_last_obs_by_indicator = _observed_latest.set_index("indicator")["lastObserved"]
_assoc_groups_by_indicator = _observed_latest.set_index("indicator")[_assoc_groups_src_col].map(_extract_group_ids)

# Ensure df's last observed column is datetime-like, then overwrite for matches
_df_ind = df[_indicator_col].astype(str)
df[_last_observed_col] = pd.to_datetime(df[_last_observed_col], utc=True, errors="coerce")
df[_last_observed_col] = _df_ind.map(_last_obs_by_indicator).combine_first(df[_last_observed_col])

# Add associatedGroups.data ids from observed_src by indicator, stored as 'Associated Groups'
if _assoc_groups_target_col in df.columns:
    df[_assoc_groups_target_col] = _df_ind.map(_assoc_groups_by_indicator).combine_first(df[_assoc_groups_target_col])
else:
    df[_assoc_groups_target_col] = _df_ind.map(_assoc_groups_by_indicator)

df

,Indicator,Last Observed,Indicator Type,Observation Yearly Count,ThreatConnect Rating,Observation Penalty Multiplier,Botnet Flag,False Positives,Partners,incidents/events,Threat Actor,CAL Score,ThreatConnect Score,HTOC Threat Score,Severity,Explanation,Associated Groups
2,103.120.116.162,2026-03-20 00:00:00+00:00,Address,51,3,0.997205,0,0,"CMS, DHA, FDA, HRSA, OS, VA",Incident:INC9385644,NaN,770,885,360,medium,[2026-03-19] Severity: medium. VT score: 6. Co...,<NA>
4,103.140.62.43,2026-03-19 00:00:00+00:00,Address,14,5,0.999233,0,0,"DHA, VA",NaN,NaN,180,590,145,low,[2026-03-19] Severity: low. VT score: 3. Conte...,Group Id: 315446
5,103.143.10.79,2026-03-20 00:00:00+00:00,Address,20,3,0.998904,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",Incident:INC9426960,NaN,710,734,632,high,[2026-03-19] Severity: high. VT score: 15. Con...,<NA>
7,103.203.59.0,2026-03-20 00:00:00+00:00,Address,285,3,0.984384,0,0,"CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS",NaN,NaN,180,368,90,low,[2026-03-19] Severity: low. VT score: 1. Conte...,<NA>
10,103.243.26.49,2026-03-20 00:00:00+00:00,Address,58,3,0.996822,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",Incident:INC9375302,NaN,180,469,351,medium,[2026-03-19] Severity: medium. VT score: 6. Co...,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1371,185.165.169.31,2026-03-19 00:00:00+00:00,Address,0,5,1.000000,0,0,NIH,NaN,NaN,180,507,354,medium,[2026-03-19] Severity: medium. VT score: 12. C...,"Group Id: 5629499540002138, Group Id: 56294995..."
1372,188.166.69.88,2026-03-19 00:00:00+00:00,Address,0,3,1.000000,0,0,IHS,Incident:INC9411178,NaN,180,590,321,medium,[2026-03-19] Severity: medium. VT score: 8. Co...,<NA>
1373,34.201.223.181,2026-03-19 00:00:00+00:00,Address,0,3,1.000000,0,0,NIH,Incident:INC9419847,NaN,160,580,70,low,[2026-03-19] Severity: low. VT score: 0. Conte...,<NA>
1510,45.134.142.194,2026-03-19 00:00:00+00:00,Address,16,5,0.999123,0,0,"CMS, DHA, HHS, OS, VA",NaN,NaN,170,428,411,medium,Severity: medium. VT score: 5. Top drivers: TC...,Group Id: 6755399444000513


In [4]:
# Filter last 24h results to indicators seen at more than one partner
last_24h_multiple_partners = df[df['Partners'].str.contains(',', na=False)]

last_24h_multiple_partners

,Indicator,Last Observed,Indicator Type,Observation Yearly Count,ThreatConnect Rating,Observation Penalty Multiplier,Botnet Flag,False Positives,Partners,incidents/events,Threat Actor,CAL Score,ThreatConnect Score,HTOC Threat Score,Severity,Explanation,Associated Groups
2,103.120.116.162,2026-03-20 00:00:00+00:00,Address,51,3,0.997205,0,0,"CMS, DHA, FDA, HRSA, OS, VA",Incident:INC9385644,NaN,770,885,360,medium,[2026-03-19] Severity: medium. VT score: 6. Co...,<NA>
4,103.140.62.43,2026-03-19 00:00:00+00:00,Address,14,5,0.999233,0,0,"DHA, VA",NaN,NaN,180,590,145,low,[2026-03-19] Severity: low. VT score: 3. Conte...,Group Id: 315446
5,103.143.10.79,2026-03-20 00:00:00+00:00,Address,20,3,0.998904,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",Incident:INC9426960,NaN,710,734,632,high,[2026-03-19] Severity: high. VT score: 15. Con...,<NA>
7,103.203.59.0,2026-03-20 00:00:00+00:00,Address,285,3,0.984384,0,0,"CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS",NaN,NaN,180,368,90,low,[2026-03-19] Severity: low. VT score: 1. Conte...,<NA>
10,103.243.26.49,2026-03-20 00:00:00+00:00,Address,58,3,0.996822,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",Incident:INC9375302,NaN,180,469,351,medium,[2026-03-19] Severity: medium. VT score: 6. Co...,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1365,144.86.173.132,2026-03-20 00:00:00+00:00,Address,4,3,0.999781,0,0,"CDC, CMS, DHA, FDA, HRSA, IHS, NIH, OS",Incident:INC9448921,NaN,380,690,199,low,[2026-03-19] Severity: low. VT score: 2. Conte...,<NA>
1366,144.86.173.179,2026-03-20 00:00:00+00:00,Address,3,3,0.999836,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS",Incident:INC9451788,NaN,300,650,199,low,[2026-03-19] Severity: low. VT score: 1. Conte...,<NA>
1368,164.82.20.53,2026-03-19 00:00:00+00:00,Address,2,4,0.999890,0,0,"CDC, DHA",NaN,"Comment Panda, Salt Typhoon, UNC5807, Volt Typ...",170,363,464,medium,[2026-03-19] Severity: medium. VT score: 6. Co...,Group Id: 6755399460002104
1510,45.134.142.194,2026-03-19 00:00:00+00:00,Address,16,5,0.999123,0,0,"CMS, DHA, HHS, OS, VA",NaN,NaN,170,428,411,medium,Severity: medium. VT score: 5. Top drivers: TC...,Group Id: 6755399444000513


In [5]:
# Filter multi-partner, last-24h indicators to VT score >= 10 based on Explanation text
vt_scores = last_24h_multiple_partners['Explanation'].str.extract(r'VT score:\s*(\d+)', expand=False)
vt_scores = pd.to_numeric(vt_scores, errors='coerce')

last_24h_multi_partners_vt15 = last_24h_multiple_partners[vt_scores >= 15]

last_24h_multi_partners_vt15

,Indicator,Last Observed,Indicator Type,Observation Yearly Count,ThreatConnect Rating,Observation Penalty Multiplier,Botnet Flag,False Positives,Partners,incidents/events,Threat Actor,CAL Score,ThreatConnect Score,HTOC Threat Score,Severity,Explanation,Associated Groups
5,103.143.10.79,2026-03-20 00:00:00+00:00,Address,20,3,0.998904,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",Incident:INC9426960,NaN,710,734,632,high,[2026-03-19] Severity: high. VT score: 15. Con...,<NA>
94,138.59.233.5,2026-03-20 00:00:00+00:00,Address,74,3,0.995945,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",Incident:INC9346798,NaN,570,615,627,high,[2026-03-19] Severity: high. VT score: 15. Con...,<NA>
96,139.19.117.131,2026-03-20 00:00:00+00:00,Address,162,3,0.991123,0,0,"CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS, VA",Incident:INC9267413,NaN,810,731,630,high,[2026-03-19] Severity: high. VT score: 15. Con...,<NA>
200,165.154.10.165,2026-03-20 00:00:00+00:00,Address,209,3,0.988548,0,0,"CDC, CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS, VA",Incident:INC9203759,NaN,390,444,621,high,[2026-03-19] Severity: high. VT score: 15. Con...,<NA>
201,165.154.173.226,2026-03-20 00:00:00+00:00,Address,183,3,0.989973,0,0,"CDC, CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS, VA",Incident:INC9235915,NaN,690,594,628,high,[2026-03-19] Severity: high. VT score: 15. Con...,<NA>
216,167.94.138.124,2026-03-20 00:00:00+00:00,Address,123,3,0.993260,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",NaN,NaN,540,613,533,high,[2026-03-19] Severity: high. VT score: 16. Con...,<NA>
260,171.25.193.235,2026-03-19 00:00:00+00:00,Address,218,5,0.988055,0,0,"CMS, FDA, HHS, HRSA, IHS, NIH, OS, VA",NaN,"Fancy Bear, Seashell Blizzard, UNC4487",210,356,578,high,[2026-03-19] Severity: high. VT score: 16. Con...,"Group Id: 6755399448000753, Group Id: 288373, ..."
316,184.105.247.252,2026-03-20 00:00:00+00:00,Address,94,3,0.994849,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",Incident:INC9332624,NaN,240,446,549,high,[2026-03-19] Severity: high. VT score: 15. Con...,<NA>
483,193.32.162.145,2026-03-20 00:00:00+00:00,Address,156,3,0.991452,0,0,"CDC, CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS, VA",Incident:INC9263763,NaN,820,789,583,high,[2026-03-19] Severity: high. VT score: 16. Con...,Group Id: 5629499579002100
535,198.235.24.194,2026-03-20 00:00:00+00:00,Address,235,3,0.987123,0,0,"CDC, CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS, VA",NaN,NaN,180,288,504,high,[2026-03-19] Severity: high. VT score: 15. Con...,<NA>


In [6]:
# Keep only high or critical indicators from the VT>=10, multi-partner, last-24h set
final_indicators = last_24h_multi_partners_vt15[last_24h_multi_partners_vt15['Severity'].isin(['high', 'critical'])]

final_indicators

,Indicator,Last Observed,Indicator Type,Observation Yearly Count,ThreatConnect Rating,Observation Penalty Multiplier,Botnet Flag,False Positives,Partners,incidents/events,Threat Actor,CAL Score,ThreatConnect Score,HTOC Threat Score,Severity,Explanation,Associated Groups
5,103.143.10.79,2026-03-20 00:00:00+00:00,Address,20,3,0.998904,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",Incident:INC9426960,NaN,710,734,632,high,[2026-03-19] Severity: high. VT score: 15. Con...,<NA>
94,138.59.233.5,2026-03-20 00:00:00+00:00,Address,74,3,0.995945,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",Incident:INC9346798,NaN,570,615,627,high,[2026-03-19] Severity: high. VT score: 15. Con...,<NA>
96,139.19.117.131,2026-03-20 00:00:00+00:00,Address,162,3,0.991123,0,0,"CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS, VA",Incident:INC9267413,NaN,810,731,630,high,[2026-03-19] Severity: high. VT score: 15. Con...,<NA>
200,165.154.10.165,2026-03-20 00:00:00+00:00,Address,209,3,0.988548,0,0,"CDC, CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS, VA",Incident:INC9203759,NaN,390,444,621,high,[2026-03-19] Severity: high. VT score: 15. Con...,<NA>
201,165.154.173.226,2026-03-20 00:00:00+00:00,Address,183,3,0.989973,0,0,"CDC, CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS, VA",Incident:INC9235915,NaN,690,594,628,high,[2026-03-19] Severity: high. VT score: 15. Con...,<NA>
216,167.94.138.124,2026-03-20 00:00:00+00:00,Address,123,3,0.993260,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",NaN,NaN,540,613,533,high,[2026-03-19] Severity: high. VT score: 16. Con...,<NA>
260,171.25.193.235,2026-03-19 00:00:00+00:00,Address,218,5,0.988055,0,0,"CMS, FDA, HHS, HRSA, IHS, NIH, OS, VA",NaN,"Fancy Bear, Seashell Blizzard, UNC4487",210,356,578,high,[2026-03-19] Severity: high. VT score: 16. Con...,"Group Id: 6755399448000753, Group Id: 288373, ..."
316,184.105.247.252,2026-03-20 00:00:00+00:00,Address,94,3,0.994849,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",Incident:INC9332624,NaN,240,446,549,high,[2026-03-19] Severity: high. VT score: 15. Con...,<NA>
483,193.32.162.145,2026-03-20 00:00:00+00:00,Address,156,3,0.991452,0,0,"CDC, CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS, VA",Incident:INC9263763,NaN,820,789,583,high,[2026-03-19] Severity: high. VT score: 16. Con...,Group Id: 5629499579002100
535,198.235.24.194,2026-03-20 00:00:00+00:00,Address,235,3,0.987123,0,0,"CDC, CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS, VA",NaN,NaN,180,288,504,high,[2026-03-19] Severity: high. VT score: 15. Con...,<NA>


In [7]:
import pandas as pd

# Helper to see if an indicator has an I&W tag
def has_iw(tags_value):
    """
    tags_value is typically a list of dicts from ThreatConnect, e.g.:
    [{'name': 'I&W'}, {'name': 'something else'}, ...]
    """
    if tags_value is None or (isinstance(tags_value, float) and pd.isna(tags_value)):
        return False

    if not isinstance(tags_value, (list, tuple)):
        return False

    for t in tags_value:
        try:
            if isinstance(t, dict):
                name = str(t.get('name', '')).strip()
            else:
                name = str(t).strip()

            if name.lower() in {"i&w", "i & w", "iw"}:
                return True
        except Exception:
            continue
    return False

# 1) Add has_iw flag to observed_src if tags.data exists
if 'tags.data' in observed_src.columns:
    observed_src['has_iw'] = observed_src['tags.data'].apply(has_iw)
else:
    observed_src['has_iw'] = False

# 2) Collapse to one flag per indicator
iw_per_indicator = (
    observed_src.groupby('indicator', dropna=False)['has_iw']
    .max()  # any True -> True
    .reset_index()
    .rename(columns={'indicator': 'Indicator', 'has_iw': 'Reported I&W?_raw'})
)

# 3) Drop ANY existing Reported I&W? variants (_x, _y, etc.)
cols_to_drop = [c for c in final_indicators.columns if c.startswith('Reported I&W?')]
final_indicators = final_indicators.drop(columns=cols_to_drop, errors='ignore')

# 4) Merge once, with a temporary raw boolean column
final_indicators = final_indicators.merge(
    iw_per_indicator,
    on='Indicator',
    how='left'
)

# 5) Convert to Yes/No, defaulting missing to 'No'
final_indicators['Reported I&W?'] = (
    final_indicators['Reported I&W?_raw']
    .fillna(False)
    .map({True: 'Yes', False: 'No'})
)

# 6) Drop the temporary raw column
final_indicators = final_indicators.drop(columns=['Reported I&W?_raw'])

final_indicators

,Indicator,Last Observed,Indicator Type,Observation Yearly Count,ThreatConnect Rating,Observation Penalty Multiplier,Botnet Flag,False Positives,Partners,incidents/events,Threat Actor,CAL Score,ThreatConnect Score,HTOC Threat Score,Severity,Explanation,Associated Groups,Reported I&W?
0,103.143.10.79,2026-03-20 00:00:00+00:00,Address,20,3,0.998904,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",Incident:INC9426960,NaN,710,734,632,high,[2026-03-19] Severity: high. VT score: 15. Con...,<NA>,No
1,138.59.233.5,2026-03-20 00:00:00+00:00,Address,74,3,0.995945,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",Incident:INC9346798,NaN,570,615,627,high,[2026-03-19] Severity: high. VT score: 15. Con...,<NA>,No
2,139.19.117.131,2026-03-20 00:00:00+00:00,Address,162,3,0.991123,0,0,"CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS, VA",Incident:INC9267413,NaN,810,731,630,high,[2026-03-19] Severity: high. VT score: 15. Con...,<NA>,No
3,165.154.10.165,2026-03-20 00:00:00+00:00,Address,209,3,0.988548,0,0,"CDC, CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS, VA",Incident:INC9203759,NaN,390,444,621,high,[2026-03-19] Severity: high. VT score: 15. Con...,<NA>,No
4,165.154.173.226,2026-03-20 00:00:00+00:00,Address,183,3,0.989973,0,0,"CDC, CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS, VA",Incident:INC9235915,NaN,690,594,628,high,[2026-03-19] Severity: high. VT score: 15. Con...,<NA>,No
5,167.94.138.124,2026-03-20 00:00:00+00:00,Address,123,3,0.993260,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",NaN,NaN,540,613,533,high,[2026-03-19] Severity: high. VT score: 16. Con...,<NA>,No
6,171.25.193.235,2026-03-19 00:00:00+00:00,Address,218,5,0.988055,0,0,"CMS, FDA, HHS, HRSA, IHS, NIH, OS, VA",NaN,"Fancy Bear, Seashell Blizzard, UNC4487",210,356,578,high,[2026-03-19] Severity: high. VT score: 16. Con...,"Group Id: 6755399448000753, Group Id: 288373, ...",Yes
7,184.105.247.252,2026-03-20 00:00:00+00:00,Address,94,3,0.994849,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",Incident:INC9332624,NaN,240,446,549,high,[2026-03-19] Severity: high. VT score: 15. Con...,<NA>,No
8,193.32.162.145,2026-03-20 00:00:00+00:00,Address,156,3,0.991452,0,0,"CDC, CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS, VA",Incident:INC9263763,NaN,820,789,583,high,[2026-03-19] Severity: high. VT score: 16. Con...,Group Id: 5629499579002100,Yes
9,198.235.24.194,2026-03-20 00:00:00+00:00,Address,235,3,0.987123,0,0,"CDC, CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS, VA",NaN,NaN,180,288,504,high,[2026-03-19] Severity: high. VT score: 15. Con...,<NA>,No


In [10]:
from datetime import datetime

# Build dated output path
today_str = datetime.today().strftime('%Y%m%d')  # e.g. 20260316
output_path = rf"Z:\HTOC\Data_Analytics\Data\Threat Assessment Scores\ThreatAssessI_W\ThreatAssessI_W_{today_str}.xlsx"

# Excel can't write timezone-aware datetimes; strip tz info before export
_dt_tz_cols = final_indicators.select_dtypes(include=["datetimetz"]).columns
for _c in _dt_tz_cols:
    final_indicators[_c] = final_indicators[_c].dt.tz_convert(None)

iw_col = "Reported I&W?"
if iw_col not in final_indicators.columns:
    raise KeyError(f"Missing required column '{iw_col}' for sheet split.")

final_iw_no = final_indicators[final_indicators[iw_col] == "No"].copy()
final_iw_yes = final_indicators[final_indicators[iw_col] == "Yes"].copy()

# Write to one workbook with two named sheets
with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
    final_iw_no.to_excel(writer, index=False, sheet_name="I&W_No")
    final_iw_yes.to_excel(writer, index=False, sheet_name="I&W_Yes")

    workbook = writer.book
    wrap_fmt = workbook.add_format({"text_wrap": True, "valign": "top"})

    # Set defaults first, then tune long text columns for each sheet
    for sheet_name, sheet_df in [("I&W_No", final_iw_no), ("I&W_Yes", final_iw_yes)]:
        worksheet = writer.sheets[sheet_name]
        worksheet.set_column(0, len(final_indicators.columns) - 1, 18)

        if "Explanation" in final_indicators.columns:
            _exp_idx = final_indicators.columns.get_loc("Explanation")
            worksheet.set_column(_exp_idx, _exp_idx, 100, wrap_fmt)

        if "Associated Groups" in final_indicators.columns:
            _ag_idx = final_indicators.columns.get_loc("Associated Groups")
            worksheet.set_column(_ag_idx, _ag_idx, 45, wrap_fmt)

output_path

'Z:\\HTOC\\Data_Analytics\\Data\\Threat Assessment Scores\\ThreatAssessI_W\\ThreatAssessI_W_20260320.xlsx'